### Lab 8.1 Tokenization

This week we will work up to creating an RNN text generator.  In today's lab you will explore different methods of text tokenization.   Here's an overview of what you will try to do.

Imagine that our entire dataset consists of the following text:

    hello world hello a b c

We would first build a vocabulary of the words in the dataset:

    0: hello
    1: world
    2: a
    3: b
    4: c

Thus the dataset can be mapped to token indices:

    0 1 0 2 3 4

Now suppose that we have defined the maximum sequence length (`seq_len`) to be 3.  We will use each possible sequence as the input to our RNN, and the next token as the target.  Here are the possible input sequences and targets:

    0 1 0 -> 2
    1 0 2 -> 3
    0 2 3 -> 4

You will build a subclass of `Dataset` to find all possible sequences for a given dataset, either at the word or character level.

The following code will download the text of Shakespeare's sonnets and read it in as one long string.

In [1]:
from torch.utils.data import Dataset

In [2]:
!wget --no-clobber "https://www.dropbox.com/scl/fi/7r68l64ijemidyb9lf80q/sonnets.txt?rlkey=udb47coatr2zbrk31hsfbr22y&dl=1" -O sonnets.txt
text = (open("sonnets.txt").read())


--2025-02-26 06:55:10--  https://www.dropbox.com/scl/fi/7r68l64ijemidyb9lf80q/sonnets.txt?rlkey=udb47coatr2zbrk31hsfbr22y&dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.5.18, 2620:100:601d:18::a27d:512
Connecting to www.dropbox.com (www.dropbox.com)|162.125.5.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://ucae23fbe2302c9212aef46e2696.dl.dropboxusercontent.com/cd/0/inline/Ck143HnJz_gmGZciKmgDy-OI9x17N4XMbJOAvRE5JUQsj9lt8UsUxbBrDuNooNq_wm2Ar6OV3cbYV4JrW-IjkcJ5SJKKZjN4mNrauTMbDHHu2q1kKfGxfQr96lIQK1jtlH2A_NemQKufR1tN-ureU3Vr/file?dl=1# [following]
--2025-02-26 06:55:11--  https://ucae23fbe2302c9212aef46e2696.dl.dropboxusercontent.com/cd/0/inline/Ck143HnJz_gmGZciKmgDy-OI9x17N4XMbJOAvRE5JUQsj9lt8UsUxbBrDuNooNq_wm2Ar6OV3cbYV4JrW-IjkcJ5SJKKZjN4mNrauTMbDHHu2q1kKfGxfQr96lIQK1jtlH2A_NemQKufR1tN-ureU3Vr/file?dl=1
Resolving ucae23fbe2302c9212aef46e2696.dl.dropboxusercontent.com (ucae23fbe2302c9212aef46e2696.dl.dropboxusercontent.com)... 162.12

In [3]:
text = text.lower()

In [4]:
print(text[:1000])

﻿i

 from fairest creatures we desire increase,
 that thereby beauty's rose might never die,
 but as the riper should by time decease,
 his tender heir might bear his memory:
 but thou, contracted to thine own bright eyes,
 feed'st thy light's flame with self-substantial fuel,
 making a famine where abundance lies,
 thy self thy foe, to thy sweet self too cruel:
 thou that art now the world's fresh ornament,
 and only herald to the gaudy spring,
 within thine own bud buriest thy content,
 and tender churl mak'st waste in niggarding:
   pity the world, or else this glutton be,
   to eat the world's due, by the grave and thee.

 ii

 when forty winters shall besiege thy brow,
 and dig deep trenches in thy beauty's field,
 thy youth's proud livery so gazed on now,
 will be a tatter'd weed of small worth held:
 then being asked, where all thy beauty lies,
 where all the treasure of thy lusty days;
 to say, within thine own deep sunken eyes,
 were an all-eating shame, and thriftless praise.

### Exercises

1. Prepare a vocabulary of the unique words in the dataset.  (For simplicity's sake you can leave the punctuation in.)

In [ ]:
words = [word for word in text.split() if word]
vocab = list(dict.fromkeys(words))

2. Now you will make a Dataset subclass that can return sequences of tokens, encoded as integers.

In [6]:
import torch
from torch.utils.data import Dataset

class WordDataset(Dataset):
  def __init__(self,text,seq_len=100):
    self.seq_len = seq_len
    self.words = [word for word in text.split() if word]
    self.vocab = list(dict.fromkeys(self.words))
    self.word2idx = {word: idx for idx, word in enumerate(self.vocab)}
    self.idx2word = {idx: word for idx, word in enumerate(self.vocab)}
    self.tokens = [self.word2idx[word] for word in self.words]

  def __len__(self):
    return len(self.tokens) - self.seq_len

  def __getitem__(self,i):
    inputs = torch.tensor(self.tokens[i : i + self.seq_len], dtype=torch.long)
    target = torch.tensor(self.tokens[i + self.seq_len], dtype=torch.long)
    return inputs, target

  def decode(self,tokens):
    if isinstance(tokens, torch.Tensor):
      tokens = tokens.tolist()
    return ' '.join([self.idx2word[token] for token in tokens])

3. Verify that your class can successfully encode and decode sequences.

In [14]:
test_text = "hello world hello a b c"
dataset = WordDataset(test_text, seq_len=3)

inputs, target = dataset[0]
decoded_input = dataset.decode(inputs)
decoded_target = dataset.idx2word[target.item()]

print(f"Input: {decoded_input}")  # Output: "hello world hello"
print(f"Target: {decoded_target}")  # Output: "a"


Input: hello world hello
Target: a


4. Do the exercise again, but this time at the character level.

In [9]:
class CharacterDataset(Dataset):
  def __init__(self,text,seq_len=100):
    self.seq_len = seq_len
    self.text = text
    self.chars = list(text)
    self.vocab = sorted(set(self.chars))
    self.char2idx = {char: idx for idx, char in enumerate(self.vocab)}
    self.idx2char = {idx: char for idx, char in enumerate(self.vocab)}
    self.tokens = [self.char2idx[char] for char in self.chars]

  def __len__(self):
    return len(self.tokens) - self.seq_len
  def __getitem__(self,i):
    inputs = torch.tensor(self.tokens[i : i + self.seq_len], dtype=torch.long)
    target = torch.tensor(self.tokens[i + self.seq_len], dtype=torch.long)
    return inputs, target

  def decode(self,tokens):
    if isinstance(tokens, torch.Tensor):
      tokens = tokens.tolist()
    return ''.join([self.idx2char[token] for token in tokens])

5. Compare the number of sequences for each tokenization method.

In [10]:
text = open("sonnets.txt").read().lower()
seq_len = 100

word_dataset = WordDataset(text, seq_len)
char_dataset = CharacterDataset(text, seq_len)

print(f"Number of sequences at word level: {len(word_dataset)}")
print(f"Number of sequences at character level: {len(char_dataset)}")

Number of sequences at word level: 17570
Number of sequences at character level: 97820


6. Optional: implement the byte pair encoding algorithm to make a Dataset class that uses word parts.